<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/mixed_precision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate

In [ ]:
!pip install -q transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading 4-bit...")
model_4bit= AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,  device_map="auto" )


print("Loading BF16...")
model_bf16 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")


print("Loading FP32...")
model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32, device_map="auto")

print("Done!")

Loading 4-bit...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Loading BF16...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading FP32...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Done!


In [ ]:
def model_size_gb(model):
  return sum(p.nbytes for p in model.parameters()) / 1e9

print(f"4-bit : {model_size_gb(model_4bit):.2f} GB")
print(f"BF16: {model_size_gb(model_bf16):.2f} GB")
print(f"FP32: {model_size_gb(model_fp32):.2f} GB")

4-bit : 0.75 GB
BF16: 2.20 GB
FP32: 4.40 GB


In [ ]:

w_fp32 = model_fp32.model.layers[0].self_attn.q_proj.weight.float()
w_bf16 = model_bf16.model.layers[0].self_attn.q_proj.weight.float()
w_4bit = model_4bit.model.layers[0].self_attn.q_proj.weight.dequantize().float()


print("First 5 weight values:")
print(f"  FP32  : {w_fp32[0, :5].tolist()}")
print(f"  BF16  : {w_bf16[0, :5].tolist()}")
print(f"  4-bit : {w_4bit[0, :5].tolist()}")


print(f"\nMax error  BF16 vs FP32 : {(w_bf16 - w_fp32).abs().max():.6f}")


First 5 weight values:
  FP32  : [-0.00148773193359375, -0.00244140625, -0.007415771484375, -0.0140380859375, -0.002899169921875]
  BF16  : [-0.00148773193359375, -0.00244140625, -0.007415771484375, -0.0140380859375, -0.002899169921875]
  4-bit : [102.0]

Max error  BF16 vs FP32 : 0.000000


In [ ]:
import bitsandbytes as bnb


layer = model_4bit.model.layers[0].self_attn.q_proj


w_4bit = bnb.functional.dequantize_4bit(
    layer.weight.data,
    layer.weight.quant_state
).float()

print(f"4-bit : {w_4bit[0, :5].tolist()}")
print(f"Max error 4bit vs FP32 : {(w_4bit - w_fp32).abs().max():.6f}")

4-bit : [-0.00250244140625, -0.00250244140625, -0.0078125, -0.014404296875, -0.00250244140625]
Max error 4bit vs FP32 : 0.066406


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


prompt = "The theory of relativity states that"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.autocast("cuda", dtype=torch.bfloat16):
  outputs = model_fp32(**inputs, labels=inputs["input_ids"])
  loss = outputs.loss

loss.backward()

grad = model_fp32.model.layers[0].self_attn.q_proj.weight.grad
print(f"Gradient dtype   : {grad.dtype}")
print(f"Gradient max val : {grad.abs().max():.8f}")
print(f"Gradient min val : {grad.abs().min():.8f}")
print(f"Loss value       : {loss.item():.4f}")


Gradient dtype   : torch.float32
Gradient max val : 0.03735352
Gradient min val : 0.00000000
Loss value       : 3.4390


expected — gradients are stored in FP32
even though the forward pass ran in BF16
this is the "master copy"
autocast computes in BF16, but stores gradients in FP32

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.net(x)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = MyModel().to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()


scaler = torch.cuda.amp.GradScaler()


x = torch.randn(32, 784).to(device)
y = torch.randint(0, 10, (32,)).to(device)


for epoch in range(5):
    optimizer.zero_grad()


    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        loss = criterion(logits, y)

    # Backprop with gradient scaling
    scaler.scale(loss).backward()

    scaler.step(optimizer)
    scaler.update()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

/tmp/ipykernel_1208/687179326.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 0, Loss: 2.381988525390625
Epoch 1, Loss: 2.27716064453125
Epoch 2, Loss: 2.1746826171875
Epoch 3, Loss: 2.075286865234375
Epoch 4, Loss: 1.9783935546875


In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

Tesla T4
True
